# Basic cleaning and exploratory data analysis (EDA)
---

## Goals
- Creating environment
- Importing libraries.
- Importing ticker data.
- Viewing columns, data and data types.
- Cleaning nan or 0 values, removing these valued rows as they are most likely non trading days and would skew the aggregating functions later on.
- Converting column data types for consistency.
---

## Creating environment
```Bash
mamba create -n stock_project python=3.12 yfinance pandas numpy seaborn matplotlib
mamba activate stock_project
```

---
## Importing libraries

In [36]:
import yfinance as yf
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

---
## Importing stock data
- Will import
    - HDFC Bank
    - ICICI Bank
    - Bank Nifty
- The time period will be 10 years

In [2]:
tickers = ['HDFCBANK.NS', 'ICICIBANK.NS', '^NSEBANK']
data = yf.download(tickers, period = "10y", auto_adjust=True)
# auto_adjust=True for handling stock splits and dividends in the source data.

[*********************100%***********************]  3 of 3 completed


---
## Quick view of the data
- Will see some rows
- Check data types of the columns

In [5]:
print("Data ->")
print(data.head())

Data ->
Price            Close                                   High               \
Ticker     HDFCBANK.NS ICICIBANK.NS      ^NSEBANK HDFCBANK.NS ICICIBANK.NS   
Date                                                                         
2016-03-21  240.374512   199.372620  15925.914062  240.776822   200.009327   
2016-03-22  242.271103   198.311447  15935.814453  242.903304   199.160381   
2016-03-23  241.236572   198.820786  15887.565430  242.087187   199.499941   
2016-03-28  240.811295   191.392609  15604.718750  242.075698   198.651005   
2016-03-29  242.259598   189.652313  15666.068359  243.569968   192.666028   

Price                            Low                                   Open  \
Ticker          ^NSEBANK HDFCBANK.NS ICICIBANK.NS      ^NSEBANK HDFCBANK.NS   
Date                                                                          
2016-03-21  15945.614028  237.305466   196.486235  15735.366478  237.477885   
2016-03-22  15967.663692  239.248025   195.764647  

In [6]:
print(data.info())

<class 'pandas.DataFrame'>
DatetimeIndex: 2470 entries, 2016-03-21 to 2026-03-19
Data columns (total 15 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   (Close, HDFCBANK.NS)    2470 non-null   float64
 1   (Close, ICICIBANK.NS)   2470 non-null   float64
 2   (Close, ^NSEBANK)       2466 non-null   float64
 3   (High, HDFCBANK.NS)     2470 non-null   float64
 4   (High, ICICIBANK.NS)    2470 non-null   float64
 5   (High, ^NSEBANK)        2466 non-null   float64
 6   (Low, HDFCBANK.NS)      2470 non-null   float64
 7   (Low, ICICIBANK.NS)     2470 non-null   float64
 8   (Low, ^NSEBANK)         2466 non-null   float64
 9   (Open, HDFCBANK.NS)     2470 non-null   float64
 10  (Open, ICICIBANK.NS)    2470 non-null   float64
 11  (Open, ^NSEBANK)        2466 non-null   float64
 12  (Volume, HDFCBANK.NS)   2470 non-null   int64  
 13  (Volume, ICICIBANK.NS)  2470 non-null   int64  
 14  (Volume, ^NSEBANK)      2466 non-

---
## Checking for nan values & 0 values

In [ ]:
print("How many nan values -> ")
print(data.isna().sum())
print("How many 0 values -> ")
print((data == 0).sum())

How many nan values -> 
Price   Ticker      
Close   HDFCBANK.NS     0
        ICICIBANK.NS    0
        ^NSEBANK        4
High    HDFCBANK.NS     0
        ICICIBANK.NS    0
        ^NSEBANK        4
Low     HDFCBANK.NS     0
        ICICIBANK.NS    0
        ^NSEBANK        4
Open    HDFCBANK.NS     0
        ICICIBANK.NS    0
        ^NSEBANK        4
Volume  HDFCBANK.NS     0
        ICICIBANK.NS    0
        ^NSEBANK        4
dtype: int64
How many 0 values -> 
Price   Ticker      
Close   HDFCBANK.NS       0
        ICICIBANK.NS      0
        ^NSEBANK          0
High    HDFCBANK.NS       0
        ICICIBANK.NS      0
        ^NSEBANK          0
Low     HDFCBANK.NS       0
        ICICIBANK.NS      0
        ^NSEBANK          0
Open    HDFCBANK.NS       0
        ICICIBANK.NS      0
        ^NSEBANK          0
Volume  HDFCBANK.NS       2
        ICICIBANK.NS      2
        ^NSEBANK        918
dtype: int64


### Insights
- So there are only 4 nan values throughout.
- There are no 0 values in the stock prices which is a good thing, but there are 918 in volume for ^NSEBANK, and 2 each for the banks. This is most likely due to these being no trading days and rather than using nan, 0 volume was represented. There is no 0 or nan in the stock prices of the banks as they are using forward filling with the close of the last trading day for getting the stock prices.

### What to do?
- Will drop the nan values and the 0 values based on NSEBANK as it is the most major instrument and if there is no volume in it then it is most likely to be a non trading day.

---
## Dropping rows with nan and 0 values

### Dropping 0 values rows

In [31]:
data_dropped = data[~(data == 0).any(axis=1)].copy()
print("How many 0 values -> ")
print((data_dropped == 0).sum())


How many 0 values -> 
Price   Ticker      
Close   HDFCBANK.NS     0
        ICICIBANK.NS    0
        ^NSEBANK        0
High    HDFCBANK.NS     0
        ICICIBANK.NS    0
        ^NSEBANK        0
Low     HDFCBANK.NS     0
        ICICIBANK.NS    0
        ^NSEBANK        0
Open    HDFCBANK.NS     0
        ICICIBANK.NS    0
        ^NSEBANK        0
Volume  HDFCBANK.NS     0
        ICICIBANK.NS    0
        ^NSEBANK        0
dtype: int64


### Dropping nan valued rows

In [33]:
data_dropped = data[~data.isna().any(axis=1)].copy()
print("How many nan values -> ")
print(data_dropped.isna().sum())

How many nan values -> 
Price   Ticker      
Close   HDFCBANK.NS     0
        ICICIBANK.NS    0
        ^NSEBANK        0
High    HDFCBANK.NS     0
        ICICIBANK.NS    0
        ^NSEBANK        0
Low     HDFCBANK.NS     0
        ICICIBANK.NS    0
        ^NSEBANK        0
Open    HDFCBANK.NS     0
        ICICIBANK.NS    0
        ^NSEBANK        0
Volume  HDFCBANK.NS     0
        ICICIBANK.NS    0
        ^NSEBANK        0
dtype: int64


In [34]:
data_dropped.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 2466 entries, 2016-03-21 to 2026-03-19
Data columns (total 15 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   (Close, HDFCBANK.NS)    2466 non-null   float64
 1   (Close, ICICIBANK.NS)   2466 non-null   float64
 2   (Close, ^NSEBANK)       2466 non-null   float64
 3   (High, HDFCBANK.NS)     2466 non-null   float64
 4   (High, ICICIBANK.NS)    2466 non-null   float64
 5   (High, ^NSEBANK)        2466 non-null   float64
 6   (Low, HDFCBANK.NS)      2466 non-null   float64
 7   (Low, ICICIBANK.NS)     2466 non-null   float64
 8   (Low, ^NSEBANK)         2466 non-null   float64
 9   (Open, HDFCBANK.NS)     2466 non-null   float64
 10  (Open, ICICIBANK.NS)    2466 non-null   float64
 11  (Open, ^NSEBANK)        2466 non-null   float64
 12  (Volume, HDFCBANK.NS)   2466 non-null   int64  
 13  (Volume, ICICIBANK.NS)  2466 non-null   int64  
 14  (Volume, ^NSEBANK)      2466 non-

---
## Converting the HDFC & ICICI bank volume to float for consistency

In [49]:
data_cleaned = data_dropped.copy()
data_cleaned['Volume'] = data_cleaned['Volume'].astype('float64').copy()
data_cleaned.info()
data_cleaned.head()

<class 'pandas.DataFrame'>
DatetimeIndex: 2466 entries, 2016-03-21 to 2026-03-19
Data columns (total 15 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   (Close, HDFCBANK.NS)    2466 non-null   float64
 1   (Close, ICICIBANK.NS)   2466 non-null   float64
 2   (Close, ^NSEBANK)       2466 non-null   float64
 3   (High, HDFCBANK.NS)     2466 non-null   float64
 4   (High, ICICIBANK.NS)    2466 non-null   float64
 5   (High, ^NSEBANK)        2466 non-null   float64
 6   (Low, HDFCBANK.NS)      2466 non-null   float64
 7   (Low, ICICIBANK.NS)     2466 non-null   float64
 8   (Low, ^NSEBANK)         2466 non-null   float64
 9   (Open, HDFCBANK.NS)     2466 non-null   float64
 10  (Open, ICICIBANK.NS)    2466 non-null   float64
 11  (Open, ^NSEBANK)        2466 non-null   float64
 12  (Volume, HDFCBANK.NS)   2466 non-null   float64
 13  (Volume, ICICIBANK.NS)  2466 non-null   float64
 14  (Volume, ^NSEBANK)      2466 non-

Price            Close                                   High               \
Ticker     HDFCBANK.NS ICICIBANK.NS      ^NSEBANK HDFCBANK.NS ICICIBANK.NS   
Date                                                                         
2016-03-21  240.374512   199.372620  15925.914062  240.776822   200.009327   
2016-03-22  242.271103   198.311447  15935.814453  242.903304   199.160381   
2016-03-23  241.236572   198.820786  15887.565430  242.087187   199.499941   
2016-03-28  240.811295   191.392609  15604.718750  242.075698   198.651005   
2016-03-29  242.259598   189.652313  15666.068359  243.569968   192.666028   

Price                            Low                                   Open  \
Ticker          ^NSEBANK HDFCBANK.NS ICICIBANK.NS      ^NSEBANK HDFCBANK.NS   
Date                                                                          
2016-03-21  15945.614028  237.305466   196.486235  15735.366478  237.477885   
2016-03-22  15967.663692  239.248025   195.764647  15778.966670  240.581406   
2016-03-23  15919.414669  239.213544   195.934401  15799.766645  242.087187   
2016-03-28  15890.515228  239.466435   190.076761  15522.019517  240.236566   
2016-03-29  15773.916718  240.064123   188.803380  15580.619155  240.811271   

Price                                      Volume                        
Ticker     ICICIBANK.NS      ^NSEBANK HDFCBANK.NS ICICIBANK.NS ^NSEBANK  
Date                                                                     
2016-03-21   196.571131  15735.366478   5318796.0   14818419.0  73600.0  
2016-03-22   199.160381  15930.264713   4120264.0   12269251.0  58600.0  
2016-03-23   196.486212  15884.065470   8464300.0   16987529.0  64100.0  
2016-03-28   198.651005  15867.565300  11270868.0   20214214.0  83500.0  
2016-03-29   189.567417  15582.918933   6733952.0   20447782.0  69500.0